# Tutorial: Preparing submission file and run evaluation

In this notebook, a step-by-step tutorial is provided for preparing the submission file for the Task B of TalentCLEF 2026 shared task. To achieve this, the data for Task B, hosted on [Zenodo](https://doi.org/10.5281/zenodo.17625261), will be downloaded; a file with the appropriate [submission format](https://talentclef.github.io/talentclef/docs/talentclef-2026/evaluation/) will be prepared, and it will be evaluated using the [task's evaluation script](https://github.com/TalentCLEF/talentclef26_evaluation_script). Additionally, the provided format is also compatible with the Codabench benchmark where the official evaluation will be done


-----------------------------
TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices across different industries.

The second edition of TalentCLEF shared task’s will be part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2026.clef-initiative.eu/), scheduled to be held in Jena, Germany, in 2026. If you are interested in registering, you can find registration form [here](https://clef-labs-registration.dipintra.it/)


<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2026/workshop/logo_clef_jena.svg" alt="CLEF2026 logo" width="150"/>

## Imports

In [1]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer, util
import subprocess

## Download Task A files

First, let's download the Task A and Task B zip files directly from Zenodo.



In [ ]:
# Download
!wget https://zenodo.org/records/18715011/files/TaskB.zip
!unzip TaskB.zip -d taskB

## Load data

Set the environement where english dev set has been extracted:

In [8]:
english_dev_path = "/content/taskB/TaskB/validation"

Load queries and corpus elements in English from the Validation folder:

In [9]:
queries = os.path.join(english_dev_path, "queries")
corpus_elements = os.path.join(english_dev_path, "corpus_elements")

Read those files:

In [10]:
queries = pd.read_csv(queries,sep="\t")
corpus_elements = pd.read_csv(corpus_elements, sep="\t")


Transform `skill_aliases` column to a list of strings:

In [11]:
import ast
corpus_elements["skill_aliases"] = corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

Generate a mapping dictionary between IDs and texts from query and corpus element strings.

In [13]:
queries_ids = queries.q_id.to_list()
queries_texts = queries.jobtitle.to_list()
map_queries = dict(zip(queries_ids,queries_texts))

Before creating a mapping dictionary of texts to corpus_ids, explode the `skill_aliases` column:

In [14]:
list_aliases_df = corpus_elements.explode("skill_aliases")
corpus_ids = list_aliases_df.c_id.to_list()
corpus_texts = list_aliases_df.skill_aliases.to_list()
map_corpus = dict(zip(corpus_texts,corpus_ids))

## Predictions

For this example, we will load a simple embedding model:

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

For this example, we will use a very simple approach. We will apply a basic embedding model and directly embed each document, allowing us to identify the documents in the corpus that are most similar to a given query.

First, encode queries and corpus elements

In [16]:
query_embeddings = model.encode(queries_texts, convert_to_tensor=True)
corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True)

Then, compute similarities

In [17]:
similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

## Prepare submission file

The submissions must follow the TREC Run File format, including headers in the output file. This means that the fle have 6 space-spearated columns per line, with following information:

- q_id: Query ID.
- Q0: A constant identifier, usually "Q0".
- doc_id: ID of the retrieved document.
- rank: Position of the document in the ranking.
- score: Relevance score assigned by the model.
- tag: Experiment name

Let's process results and prepare output file. In this tutorial, we will only consider 20 relevant corpus per query for simplicity.

In [18]:
import numpy as np
results = []
results_name = []

for q_idx, q_id in enumerate(queries_ids):
    sorted_indices = np.argsort(-similarities[q_idx])
    used_doc_ids = set()
    rank_counter = 0
    for c_idx in sorted_indices[:20]:  # Consider 20 elements for simplicity.
        doc_id = corpus_ids[c_idx]
        # If doc_id was already processed, go to the next one.
        if doc_id in used_doc_ids:
            continue
        used_doc_ids.add(doc_id)
        rank_counter += 1

        query_name = map_queries[q_id]
        doc_name = corpus_texts[c_idx]
        score = similarities[q_idx, c_idx]

        results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} baseline_model")
        results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} baseline_model")

The output list have the expected structure:

In [ ]:
results[0:2]

Then, save the list as a trec/txt file:

In [20]:
with open("evaluation_test_en.trec", "w", encoding="utf-8") as f:
    f.write("\n".join(results))

## Evaluation

For the evaluation, we will use the official [TalentCLEF 2026 evaluation script](https://github.com/TalentCLEF/talentclef26_evaluation_script), which uses the Ranx library under the hood.

First, clone the repo and install the requirements file:

In [21]:
!git clone https://github.com/TalentCLEF/talentclef26_evaluation_script.git
!pip install -r /content/talentclef26_evaluation_script/requirements.txt


Then, select the Qrels file and the Run file to perform the evaluation.


In [22]:
qrels_file = "/content/taskB/TaskB/validation/qrels.tsv"
run_file = "/content/evaluation_test_en.trec"

Some examples on how to use the evaluation script for different scenarios is shown in the [repo README.md](https://github.com/TalentCLEF/talentclef26_evaluation_script/blob/main/README.md#examples).

In this notebook, we've been working only with english data from Task A dev set, so `--lang-mode`will be _en_, and `--task` will be _A_.

In [ ]:
command = ["python", "/content/talentclef26_evaluation_script/talentclef_evaluate.py", "--task", "B", "--qrels", qrels_file, "--run", run_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)